# Etapa 1: Obtención y Exploración del Dataset N-BaIoT (9 Dispositivos IoT)
En concordancia con la metodología del informe CINF200, este notebook ejecuta:
1. Inventario real de los **89 archivos CSV** de los 9 dispositivos comerciales del dataset **N-BaIoT** (`data_summary.csv` y `device_info.csv`).
2. Análisis de la distribución y severo desbalance entre tráfico **Benigno (Clase 0)** y ataques de las botnets **Mirai** y **BASHLITE/gafgyt (Clase 1)**.
3. Inspección estadística de las **115 características** extraídas por ventanas temporales amortiguadas (`L5, L3, L1, L0.1, L0.01`) y detección del artefacto de marca de tiempo Unix en `HH_jit_*_mean`.

In [1]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 120)

data_dir = os.path.abspath("./data")
devices_df = pd.read_csv(os.path.join(data_dir, "device_info.csv"))
summary_df = pd.read_csv(os.path.join(data_dir, "data_summary.csv"))
summary_df.columns = [c.strip() for c in summary_df.columns]

print("=== DISPOSITIVOS IoT EN N-BaIoT ===")
print(devices_df.to_string(index=False))
print(f"\nTotal de archivos CSV en el dataset: {len(summary_df)}")
print(f"Total de instancias (filas) en los 9 dispositivos: {summary_df['Data Count'].sum():,}")

=== DISPOSITIVOS IoT EN N-BaIoT ===
 DeviceID                               DeviceName
        1                         Danmini_Doorbell
        2                        Ecobee_Thermostat
        3                           Ennio_Doorbell
        4             Philips_B120N10_Baby_Monitor
        5        Provision_PT_737E_Security_Camera
        6         Provision_PT_838_Security_Camera
        7                Samsung_SNH_1011_N_Webcam
        8 SimpleHome_XCS7_1002_WHT_Security_Camera
        9 SimpleHome_XCS7_1003_WHT_Security_Camera

Total de archivos CSV en el dataset: 89
Total de instancias (filas) en los 9 dispositivos: 7,062,606


## 1. Análisis de Desbalance de Clases por Dispositivo y Botnet
A continuación cuantificamos el volumen real de tráfico benigno versus tráfico malicioso (BASHLITE y Mirai) por cada uno de los 9 dispositivos comerciales.

In [2]:
summary_df["DeviceID"] = summary_df["File Name"].apply(lambda x: int(x.split(".")[0]))
summary_df["Categoria"] = summary_df["File Name"].apply(
    lambda x: "Benigno" if "benign" in x else ("Mirai" if "mirai" in x else "BASHLITE")
)

pivot_counts = summary_df.pivot_table(
    index="DeviceID", columns="Categoria", values="Data Count", aggfunc="sum", fill_value=0
).merge(devices_df, on="DeviceID")
pivot_counts["Total_Ataque"] = pivot_counts["BASHLITE"] + pivot_counts.get("Mirai", 0)
pivot_counts["Ratio_Ataque_vs_Benigno"] = (pivot_counts["Total_Ataque"] / pivot_counts["Benigno"]).round(2)

cols_order = ["DeviceID", "DeviceName", "Benigno", "BASHLITE", "Mirai", "Total_Ataque", "Ratio_Ataque_vs_Benigno"]
print("=== DISTRIBUCIÓN DE TRÁFICO POR DISPOSITIVO (DESBALANCE REAL) ===")
print(pivot_counts[cols_order].to_string(index=False))

=== DISTRIBUCIÓN DE TRÁFICO POR DISPOSITIVO (DESBALANCE REAL) ===
 DeviceID                               DeviceName  Benigno  BASHLITE  Mirai  Total_Ataque  Ratio_Ataque_vs_Benigno
        1                         Danmini_Doorbell    49548    316650 652100        968750                    19.55
        2                        Ecobee_Thermostat    13113    310630 512133        822763                    62.74
        3                           Ennio_Doorbell    39100    316400      0        316400                     8.09
        4             Philips_B120N10_Baby_Monitor   175240    312723 610714        923437                     5.27
        5        Provision_PT_737E_Security_Camera    62154    330096 436010        766106                    12.33
        6         Provision_PT_838_Security_Camera    98514    309040 429337        738377                     7.50
        7                Samsung_SNH_1011_N_Webcam    52150    323072      0        323072                     6.20
      

## 2. Verificación de Calidad de Datos y Artefacto `HH_jit` en Dispositivo 1 (*Danmini Doorbell*)
Cargamos muestras reales de tráfico benigno (`1.benign.csv`) y de ataque (`1.mirai.udp.csv`) para verificar ausencia de valores nulos y observar el comportamiento de las 115 características.

In [3]:
df_benign = pd.read_csv(os.path.join(data_dir, "1.benign.csv"), nrows=5000)
df_mirai = pd.read_csv(os.path.join(data_dir, "1.mirai.udp.csv"), nrows=5000)

print(f"Dimensiones Benigno (muestra): {df_benign.shape} | Nulos totales: {df_benign.isnull().sum().sum()}")
print(f"Dimensiones Mirai UDP (muestra): {df_mirai.shape} | Nulos totales: {df_mirai.isnull().sum().sum()}")

cols_preview = ["MI_dir_L0.01_weight", "H_L0.1_weight", "HH_L0.01_magnitude", "HpHp_L0.1_weight", "HH_jit_L0.1_mean"]
print("\n=== ESTADÍSTICAS DESCRIPTIVAS: TRÁFICO BENIGNO (Dispositivo 1) ===")
print(df_benign[cols_preview].describe().round(3))

print("\n=== ESTADÍSTICAS DESCRIPTIVAS: ATAQUE MIRAI UDP (Dispositivo 1) ===")
print(df_mirai[cols_preview].describe().round(3))

Dimensiones Benigno (muestra): (5000, 115) | Nulos totales: 0
Dimensiones Mirai UDP (muestra): (5000, 115) | Nulos totales: 0

=== ESTADÍSTICAS DESCRIPTIVAS: TRÁFICO BENIGNO (Dispositivo 1) ===
       MI_dir_L0.01_weight  H_L0.1_weight  HH_L0.01_magnitude  HpHp_L0.1_weight  HH_jit_L0.1_mean
count             5000.000       5000.000            5000.000          5000.000      5.000000e+03
mean                30.125          4.311             131.168             1.152      7.479234e+06
std                  2.485          1.717              19.487             0.240      9.812422e+07
min                  1.000          1.000              60.000             1.000      6.271000e+00
25%                 28.703          2.880             109.896             1.072      1.919400e+01
50%                 30.243          4.233             144.250             1.072      3.901600e+01
75%                 31.634          5.612             144.250             1.142      3.903100e+01
max                 40